# v12 R: Hindi-only fix + the lesion tests on the SECOND related model (seed 1)
- **Hindi-only fix:** in v11 A the Hindi-only model's self was found on Hindi, Chinese and Arabic text, but that model only ever saw Hindi. Redone with the self found on Hindi only (phases 10, 30, 45, 60). Result: `analysis/v11/static_hindi_fix/`.
- **Seed 1 of the related run** (`v8_indic_continual_s1`, an independently trained model): sizes (phases 5 to 16), count-matched lesions without training (phases 10 to 15), then lesion + train the next language for 1,000 steps (start points 10 to 15; training seed 0 first, then seed 1).
  Results: `analysis/v11/sizes_s1/`, `static_s1/`, `train_next_s1/`.

Progress: `tail -f logs/v12_R_seed1.log`

In [ ]:
import os, sys, glob, json, time, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
GPU = os.environ.get("CML_GPU", "0"); LOG = "v12_R_seed1"
REPO = "/workspace/shreshth"; os.chdir(REPO); sys.path.insert(0, REPO)
from IPython.display import Image, display
from huggingface_hub import HfApi, hf_hub_download
FILES, HF_REPO = [], None
for attempt in range(5):
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; FILES = api.list_repo_files(HF_REPO); break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)
if not FILES:
    print("could not list the HF repo: only checkpoints already on this machine will be used")

def have(run, ph):
    return bool(glob.glob(f"runs/{run}/ckpt/p{ph:03d}_*.pt"))

def get(run, phases):
    """download what is missing; returns the phases that are on this machine afterwards"""
    ok = []
    for ph in phases:
        if not have(run, ph):
            name = [f for f in FILES if f.startswith(f"{run}/ckpt/p{ph:03d}_")]
            if name:
                try:
                    hf_hub_download(HF_REPO, name[0], local_dir="runs")
                except Exception as e:
                    print("download failed:", run, ph, type(e).__name__, e)
        if have(run, ph):
            ok.append(ph)
        else:
            print("not available (not here, not on HF):", run, ph)
    return ok

def run(langset, script, *cmd):
    """one call of a script; output shown here AND appended to logs/<LOG>.log (watch with: tail -f logs/<LOG>.log)"""
    env = {**os.environ, "CML_LANGSET": langset}; os.makedirs("logs", exist_ok=True)
    p = subprocess.Popen([sys.executable, "-u", script, "--gpu", GPU, *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    with open(f"logs/{LOG}.log", "a") as log:
        for line in p.stdout:
            print(line, end=""); log.write(line); log.flush()
    p.wait(); print("exit code", p.returncode)

def summary(script, part, out=None):
    cmd = [sys.executable, script, "--summarize", "--part", part] + (["--out", out] if out else [])
    r = subprocess.run(cmd, capture_output=True, text=True); print(r.stdout[-14000:]); print(r.stderr[-3000:])

def show(*paths):
    for f in paths:
        if os.path.exists(f): display(Image(f))

DIST, HIN, MIX, REL, REL1 = "v4b_continual_s0", "v4b_only_hi_s0", "v6b_mixed_s0", "v8_indic_continual_s0", "v8_indic_continual_s1"
print("GPU", GPU)

In [ ]:
f = "analysis/v11/static_hindi_fix/static.csv"
if os.path.exists(f): os.remove(f)                                  # this part has no resume: start clean
ok = get(HIN, [10, 30, 45, 60])
if ok:
    run("distant", "core_v11.py", "--part", "static", "--out", "static_hindi_fix", "--family", "hindi", "--self_langs", "hi", "--eval_langs", "hi",
        "--model", f"hindi={HIN}:" + ",".join(str(p) for p in ok))
summary("core_v11.py", "static", "static_hindi_fix")

In [ ]:
for f in ["analysis/v11/sizes_s1/sizes.csv", "analysis/v11/static_s1/static.csv"]:
    if os.path.exists(f): os.remove(f)
ok = get(REL1, list(range(5, 17)))
if ok:
    run("indic", "core_v11.py", "--part", "sizes", "--out", "sizes_s1", "--family", "related_s1", "--model", f"related_s1={REL1}:" + ",".join(str(p) for p in ok))
summary("core_v11.py", "sizes", "sizes_s1")
ok = get(REL1, [10, 11, 12, 13, 14, 15])
if ok:
    run("indic", "core_v11.py", "--part", "static", "--out", "static_s1", "--family", "related_s1", "--model", f"related_s1={REL1}:" + ",".join(str(p) for p in ok))
summary("core_v11.py", "static", "static_s1")

In [ ]:
for seeds in ["0", "1"]:
    for ph in get(REL1, [10, 11, 12, 13, 14, 15]):
        run("indic", "core_v11.py", "--part", "train", "--task", "next", "--mode", "lesion", "--out", "train_next_s1", "--family", "related_s1",
            "--model", f"related_s1_{ph}={REL1}:{ph}", "--seeds", seeds)
    summary("core_v11.py", "train", "train_next_s1")
show("analysis/v11/train_next_s1/curves.png")